In [5]:
# # Linear regression 

# # x - price 
# # y - date

# import datetime
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from pandas_datareader import data
# from sklearn.metrics import mean_squared_error

# tickers = ['AAPL']

# start_date = '2019-01-01'

# today = datetime.date.today()
# yesterday = today - datetime.timedelta(days=1)
# end_date = yesterday

# # User pandas_reader.data.DataReader to load data
# panel_data = data.DataReader(tickers, 'yahoo', start_date, end_date)


# # df = panel_data['Close']
# df = panel_data
# df.head()

Attributes,Adj Close,Close,High,Low,Open,Volume
Symbols,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2018-12-31,38.282612,39.435001,39.840000,39.119999,39.632500,140014000.0
2019-01-02,38.326294,39.480000,39.712502,38.557499,38.722500,148158800.0
2019-01-03,34.508713,35.547501,36.430000,35.500000,35.994999,365248800.0
2019-01-04,35.981865,37.064999,37.137501,35.950001,36.132500,234428400.0
2019-01-07,35.901775,36.982498,37.207500,36.474998,37.174999,219111200.0


In [44]:
# from sklearn.linear_model import LinearRegression
# from fastai.structured import  add_datepart

# def linear_regression(ticker):
    
#     #setting index as date values
#     df['Date'] = pd.to_datetime(df.Date,format='%Y-%m-%d')
#     df.index = df['Date']

#     #sorting
#     data = df.sort_index(ascending=True, axis=0)

#     #creating a separate dataset
#     new_data = pd.DataFrame(index=range(0,len(df)),columns=['Date', 'Close'])

#     for i in range(0,len(data)):
#         new_data['Date'][i] = data['Date'][i]
#         new_data['Close'][i] = data['Close'][i]

#     #create features
#     add_datepart(new_data, 'Date')
#     new_data.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp

#     new_data['mon_fri'] = 0
#     for i in range(0,len(new_data)):
#         if (new_data['Dayofweek'][i] == 0 or new_data['Dayofweek'][i] == 4):
#             new_data['mon_fri'][i] = 1
#         else:
#             new_data['mon_fri'][i] = 0

#     train = new_data[:987]
#     valid = new_data[987:]

#     x_train = train.drop('Close', axis=1)
#     y_train = train['Close']
#     x_valid = valid.drop('Close', axis=1)
#     y_valid = valid['Close']

#     model = LinearRegression()
#     model.fit(x_train,y_train)
    
#     #make predictions and find the rmse
#     preds = model.predict(x_valid)
#     rms=np.sqrt(np.mean(np.power((np.array(y_valid)-np.array(preds)),2)))
#     rms

#     #plot
#     valid['Predictions'] = 0
#     valid['Predictions'] = preds

#     valid.index = new_data[987:].index
#     train.index = new_data[:987].index

#     plt.plot(train['Close'])
#     plt.plot(valid[['Close', 'Predictions']])

#     return price 

ModuleNotFoundError: No module named 'fastai.structured'

In [45]:
!pip install fastai.structured

ERROR: Could not find a version that satisfies the requirement fastai.structured
ERROR: No matching distribution found for fastai.structured


In [267]:
test = linear_regression("VXRT")
test

UnboundLocalError: local variable 'data' referenced before assignment

In [268]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas_datareader import data
from sklearn.metrics import mean_squared_error
from fastai.tabular.all import *
import datetime

def linear_regression(stockname):
    global data
    tickers = [stockname]

    start_date = '2019-01-01'
    end_date = datetime.date.today() 

    # User pandas_reader.data.DataReader to load data
    panel_data = data.DataReader(tickers,'yahoo', start_date, end_date)

    # df = panel_data['Close']
    df = panel_data

    panel_data["Date"] = panel_data.index
    panel_data["Date"] = pd.to_datetime(panel_data['Date'])

    df['Date'] = pd.to_datetime(df.Date,format='%Y-%m-%d')
    df.index = df['Date']

    #sorting
    data = df.sort_index(ascending=True, axis=0)

    #creating a separate dataset
    new_data = pd.DataFrame(index=range(0,len(df)),columns=['Date', 'Close'])



    for i in range(0,len(data)):
        new_data['Date'][i] = data['Date'][i]
        new_data['Close'][i] = data['Close'].iloc[i][stockname]

    #create features
    add_datepart(new_data, 'Date')
    new_data.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp


    new_data['mon_fri'] = 0
    new_data
    for i in range(0,len(new_data)):
        if (new_data['Dayofweek'][i] == 0 or new_data['Dayofweek'][i] == 4):
            new_data.at[i,'mon_fri'] = 1
        else:
            new_data.at[i,'mon_fri'] = 0


    end = int(len(new_data) * 0.8)

    train = new_data[:end]
    valid = new_data[end:]

    valid

    x_train = train.drop('Close', axis=1)
    y_train = train['Close']
    x_valid = valid.drop('Close', axis=1)
    y_valid = valid['Close']

    model = LinearRegression()
    model.fit(x_train,y_train)


    #make predictions and find the rmse
    preds = model.predict(x_valid)
    rms=np.sqrt(np.mean(np.power((np.array(y_valid)-np.array(preds)),2)))


    yst_price = panel_data.iloc[-2]["Close"][stockname]

    tdy = pd.DataFrame([datetime.date.today()],columns=["Date"])

    add_datepart(tdy, 'Date')
    tdy.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp
    tdy['mon_fri'] = 1
    if (tdy['Dayofweek'][0] == 0 or tdy['Dayofweek'][0] == 4):
        tdy['mon_fri'] = 1
    else:
        tdy['mon_fri'] = 0


    tdy_preds = model.predict(tdy)
    tdy_preds
    json_result = {'today_price': tdy_preds, 'yesterday_price':yst_price, 'rmse': rms}
    return json_result

In [269]:
test = linear_regression("VXRT")
test

{'today_price': array([9.45888358]),
 'yesterday_price': 6.889999866485596,
 'rmse': 1.2328488081248794}